## Pydantic
- pydantic.BaseModel을 상속받는 클래스로 모델 생성

In [6]:
from pydantic import BaseModel

# 문서 한개를 담을 수 있는 클래스 생성 (타입 체크 필요)
class Document(BaseModel):
    # __init__을 생략해도, BaseModel 이 아래 필드 선언을 읽고, 자동으로 생성.    
    doc_id: str
    title: str
    version: int
    security_level: str

# 객체 생성
doc = Document(
    doc_id = "DOC-HR-001",
    title = "여비 규정",
    version = "1.0", # 적당히 알아서 타입 변환 가능
    security_level = "사내 공개",
)

print(doc) # __repr__도 자동으로 정의(클래스 상속)해준다.
print(doc.title)

doc_id='DOC-HR-001' title='여비 규정' version=1 security_level='사내 공개'
여비 규정


In [49]:
from pydantic import Field
from datetime import date

# 제약이 붙은 문서 모델
class Document2(BaseModel):
    # Field
    doc_id: str = Field(
        ...,
        pattern = r"DOC-[A-Z]{2, 4}-\d{3}$",
        description = "문서 고유 번호"
    )
    title: str = Field(
        min_length = 1,
        max_length = 200,
    )
    version: str = Field(..., pattern = r"^\d+\.\d+$") # "2.0"
    security_level: str
    expiry_date: date | None = None # 여러 타입
    tags: list[str] = Field(default_factory=list)

doc2 = Document2(
    doc_id = "DOC-HR-002",
    title = "출장비 규정",
    version = "1.2",
    security_level = "사내배포",
    tags=['aaa', 'bbb']
)
print(doc2)
print(doc2.tags)


doc_id='DOC-HR-002' title='출장비 규정' version='1.2' security_level='사내배포' expiry_date=None tags=[]
[]


In [ ]:
from enum import Enum
from typing import Literal

#enum 은 값 변경 안하는 상수들의 집합 보통 Enum의 인스턴스 변수는 전체 대문자
class SecurityLevel(str, Enum):
    # str과 함께 Enum을 상속받아 구현하며, 문자열처럼 사용할 수 있어 JSON변환이 편하다.
    PUBLIC = "공개"
    INTERNAL = "사내공개"
    CONFIDENTIAL = "대외비"
    SECRET = "기밀"

class Document3(BaseModel):
    doc_id: str
    security_level: SecurityLevel # Enum 타입으로 지정
    sec_level: Literal["공개", "사내공개", "대외비", "기밀"]

# doc3 = Document3(doc_id="DOC-1", securityLevel="대외비")
doc3 = Document3(doc_id="DOC-1", security_level= SecurityLevel.CONFIDENTIAL, sec_level = "대외비")

print(doc3)

doc_id='DOC-1' security_level=<SecurityLevel.CONFIDENTIAL: '대외비'> sec_level='대외비'


In [ ]:
class Document4(BaseModel):
    doc_id: str
    title: str
    effective_date : date
    tags: list[str] = Field(default_factory=list)

doc4 = Document4(
    doc_id = "DOC-HR-004",
    title = "국내 출장 여비 규정",
    effective_date = date(2026, 9, 7),
    tags = ["인사", "출장", "여비"]
)
print(doc4)

# 모델(파이썬 객체) -> 딕셔너리
d = doc.model_dump()
print(d)
# 모델 -> JSON
j = doc.model_dump_json(indent=2)
print(j)

doc_id='DOC-HR-004' title='국내 출장 여비 규정' effective_date=datetime.date(2026, 9, 7) tags=['인사', '출장', '여비']
{'doc_id': 'DOC-HR-001', 'title': '여비 규정', 'version': 1, 'security_level': '사내 공개'}
{
  "doc_id": "DOC-HR-001",
  "title": "여비 규정",
  "version": 1,
  "security_level": "사내 공개"
}


In [48]:
# 넘어온 딕셔너리/JSON -> 모델
data = {
  "doc_id" : "DOC-SEC-002",
  "title" : "정보 보안 지침",
  "effective_date" : "2026-09-07",
  "tags" : ["보안"]
}
doc_data = Document4.model_validate(data)
print(doc_data)
print(type(doc_data))

data_json= '{"doc_id": "DOC-SEC-002","title" : "정보 보안 지침","effective_date": "2026-09-07", "tags": ["보안"]}'
doc_json = Document4.model_validate_json(data_json)
print(doc_json)
print(type(doc_json))

doc_id='DOC-SEC-002' title='정보 보안 지침' effective_date=datetime.date(2026, 9, 7) tags=['보안']
<class '__main__.Document4'>
doc_id='DOC-SEC-002' title='정보 보안 지침' effective_date=datetime.date(2026, 9, 7) tags=['보안']
<class '__main__.Document4'>


In [ ]:
from pydantic import BaseModel, Field, field_validator
from datetime import date, datetime
from enum import Enum

# 보안 등급 : 여러파일에서 사용하므로 enum으로 고정
class SecurityLevel(str, Enum):
    PUBLIC = "공개"
    INTERNAL = "사내공개"
    CONFIDENTIAL = "대외비"
    SECRET = "기밀"

# 요청 모델 : 문서 등록 요청 - 클라이언트(사용자)가 보내는 것
class DocumentCreate(BaseModel):
    doc_id: str = Field(..., pattern=r"DOC-[A-Z]{2,4}-\d{3}$", description="문서 고유 번호 (예:DOC-HR-001)")
    title: str = Field(..., min_length=1, max_length=200) # 문서 제목
    version: str = Field(..., pattern=r"^\d+\.\d+$",description="예:1.0") #문서 버전
    department: str = Field(..., min_length=1, max_length=50) # 문서의 소속 부서
    security_level: SecurityLevel # 보안 레벨
    effective_date: date # 활성화 날짜
    expiry_date: data | None = None # 유효기간 
    
    # doc_id가 소문자로 들어와도 대문자로 저장하는 처리
    @field_validator("doc_id")
    @classmethod
    def upper_doc_id(cls, v):
        return v.upper() if isinstance(v, str) else v

    # title, department 의 앞뒤 공백을 제거한다 (여러 필드 지정 가능)
    @field_validator("title", "department")j
    @classmethod
    def strip_text(cls, v:str) -> str:
        return v.strip()



# 응답 모델 : 문서 응답 - 서버가 돌려주는 데이터 형태 (요청에는 없던 값이 추가될수 있다.)
class DocumentOut(BaseModel):
    id: int                        # 서버에서 사용될 데이터의 고유 번호
    doc_id: str
    title: str
    version: str
    department: str
    security_level: str
    effective_level: date
    expiry_date: date | None
    is_latest: bool
    chunk_count: int = Field(ge=0) # 임베딩된 조각 수
    create_at: datetime